# Tiered Recommendation System — Simulation Analysis

**Purpose:** End-to-end analysis of the Monte Carlo simulation with tiered recommendation strategies, validated against CVS 10-K financial benchmarks.

**Data Sources:**
- `data/results/simulation/summary.parquet` — Aggregated metrics across MC runs
- `data/results/simulation/run_*.parquet` — Per-run epoch-level metrics
- `data/results/simulation/tier_transitions.parquet` — Tier migration events
- `data/results/simulation/breakout_results.parquet` — Breakout promotions
- `data/model/product_tiers.parquet` — Product tier assignments
- `data/model/elasticity.parquet` — Price elasticity estimates
- `data/model/breakout_candidates.parquet` — Breakout candidate profiles

| Section | Topic |
|---------|-------|
| [1](#1) | Executive Summary |
| [2](#2) | Product Tier Distribution |
| [3](#3) | Revenue Convergence |
| [4](#4) | Price Elasticity Landscape |
| [5](#5) | Breakout Candidates |
| [6](#6) | Comparison to CVS 10-K |
| [7](#7) | Recommendation for Production |

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)

# ── Paths ──────────────────────────────────────────────────────────
ROOT = Path("..").resolve()
SIM_DIR = ROOT / "data" / "results" / "simulation"
MODEL_DIR = ROOT / "data" / "model"
WS = SIM_DIR / "workspace"

# ── Load simulation data ──────────────────────────────────────────
summary = pd.read_parquet(SIM_DIR / "summary.parquet")
run_files = sorted(SIM_DIR.glob("run_*.parquet"))
runs = {int(f.stem.split("_")[1]): pd.read_parquet(f) for f in run_files}

with open(SIM_DIR / "parameters.json") as f:
    params = json.load(f)
with open(SIM_DIR / "convergence.json") as f:
    convergence = json.load(f)

transitions = pd.read_parquet(SIM_DIR / "tier_transitions.parquet")
breakout_results = pd.read_parquet(SIM_DIR / "breakout_results.parquet")

# ── Load model artifacts ──────────────────────────────────────────
elasticity = pd.read_parquet(MODEL_DIR / "elasticity.parquet")
breakout_candidates = pd.read_parquet(MODEL_DIR / "breakout_candidates.parquet")

# Product tiers (handle duplicate columns from elasticity merge)
_pf = pq.ParquetFile(MODEL_DIR / "product_tiers.parquet")
_tbl = _pf.read()
_seen, _keep = {}, []
for i, name in enumerate(_tbl.column_names):
    if name not in _seen:
        _seen[name] = i
        _keep.append(i)
tiers_df = _tbl.select(_keep).to_pandas()
del _pf, _tbl, _seen, _keep

# ── Load workspace arrays ─────────────────────────────────────────
tier_array = np.load(WS / "product_tier_array.npy")
product_prices = np.load(WS / "product_prices.npy")
breakout_pids = np.load(WS / "breakout_pids.npy")
breakout_discounts = np.load(WS / "breakout_discounts.npy")

with open(WS / "product_lookup.json") as f:
    product_lookup = json.load(f)

# ── Constants ─────────────────────────────────────────────────────
SIM_CUSTOMERS = params["max_customer_id"] - 1
TARGET_CUSTOMERS = 10_000_000
SCALE = TARGET_CUSTOMERS / SIM_CUSTOMERS
WEEKLY_TARGET = params["weekly_revenue_target"]
NUM_RUNS = params["total_runs"]
NUM_EPOCHS = params["num_epochs"]

last = summary.iloc[-1]

print(f"Loaded {len(summary)} epochs x {NUM_RUNS} runs")
print(f"Simulation customers: {SIM_CUSTOMERS:,}")
print(f"Scale factor to 10M: {SCALE:,.0f}x")

<a id='1'></a>
## 1. Executive Summary

The tiered recommendation model segments 12,000 products into revenue-based tiers and applies differentiated coupon strategies:
- **Tier 1** (core drivers): always in basket, personalized discount by price sensitivity
- **Tier 2** (discount-responsive): top personalized coupon offers
- **Tier 4** (long tail / breakout candidates): exploratory offers at estimated discount

The Monte Carlo simulation runs multiple replications of a 40-week feedback loop (recommend → purchase → retrain → repeat) to estimate steady-state performance.

In [ ]:
# ── Executive metrics ─────────────────────────────────────────────
scaled_weekly_net = last["mean_net_revenue"] * SCALE
scaled_annual = scaled_weekly_net * 52
discount_rate = last["mean_discount_cost"] / last["mean_total_revenue"]
gross_margin_after = 0.29 - discount_rate  # CVS front store ~29%
hit5 = last["mean_hit_rate_at_5"]
hit2 = last["mean_hit_rate_at_2"]
coverage = last["mean_catalog_coverage"]
active = last["mean_active_customer_pct"]
coupons_per = last["mean_mean_coupons_per_customer"]
breakout_n = last["mean_breakout_success_count"]

# Naive baseline: epoch 1 represents the model before any retraining
first = summary.iloc[0]
naive_weekly = first["mean_net_revenue"] * SCALE
naive_annual = naive_weekly * 52
naive_disc = first["mean_discount_cost"] / first["mean_total_revenue"]
naive_hit5 = first["mean_hit_rate_at_5"]
naive_coverage = first["mean_catalog_coverage"]

comparison = pd.DataFrame({
    "Metric": [
        "Annual Revenue (scaled to 10M)",
        "Weekly Net Revenue (scaled)",
        "Hit Rate @5",
        "Catalog Coverage",
        "Discount Rate",
        "Breakout Successes",
    ],
    "Epoch 1 (Naive)": [
        f"${naive_annual / 1e9:.2f}B",
        f"${naive_weekly / 1e6:.1f}M",
        f"{naive_hit5:.1%}",
        f"{naive_coverage:.3%}",
        f"{naive_disc:.1%}",
        "0",
    ],
    f"Epoch {NUM_EPOCHS} (Tiered)": [
        f"${scaled_annual / 1e9:.2f}B",
        f"${scaled_weekly_net / 1e6:.1f}M",
        f"{hit5:.1%}",
        f"{coverage:.3%}",
        f"{discount_rate:.1%}",
        f"{breakout_n:.0f}",
    ],
    "Delta": [
        f"{(scaled_annual - naive_annual) / naive_annual:+.1%}",
        f"{(scaled_weekly_net - naive_weekly) / naive_weekly:+.1%}",
        f"{hit5 - naive_hit5:+.1%}",
        f"{coverage - naive_coverage:+.3%}",
        f"{discount_rate - naive_disc:+.1%}",
        f"+{breakout_n:.0f}",
    ],
})

print("=" * 80)
print("EXECUTIVE SUMMARY: NAIVE vs TIERED MODEL")
print("=" * 80)
display(comparison.style.hide(axis="index"))

print(f"\nFinding: Over {NUM_EPOCHS} simulated weeks, the tiered model converges to")
print(f"${scaled_annual / 1e9:.2f}B annualized revenue (target: ~$2.97B), with a")
print(f"{discount_rate:.1%} discount rate preserving an estimated {gross_margin_after:.1%} gross margin.")
print(f"{breakout_n:.0f} of {len(breakout_pids)} breakout candidates were promoted to Tier 2.")

<a id='2'></a>
## 2. Product Tier Distribution

In [ ]:
# ── 2a. Product count per tier ────────────────────────────────────
tier_counts = tiers_df["tier"].value_counts().sort_index()
tier_labels = {1: "Tier 1\n(Core Drivers)", 2: "Tier 2\n(Discount-Responsive)",
               3: "Tier 3\n(Organic Sellers)", 4: "Tier 4\n(Long Tail)"}
tier_colors = {1: "#e74c3c", 2: "#f39c12", 3: "#27ae60", 4: "#3498db"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: product count
bars = axes[0].bar(
    [tier_labels.get(t, f"Tier {t}") for t in tier_counts.index],
    tier_counts.values,
    color=[tier_colors.get(t, "#95a5a6") for t in tier_counts.index],
    edgecolor="white", linewidth=1.5,
)
for bar, val in zip(bars, tier_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f"{val:,}", ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Product Count by Tier", fontweight="bold")
axes[0].set_ylabel("Number of Products")

# Pie chart: revenue share
tier_rev = tiers_df.groupby("tier")["total_revenue"].sum().sort_index()
total_rev = tier_rev.sum()
wedge_labels = [f"{tier_labels.get(t, f'Tier {t}')}\n{tier_rev[t]/total_rev:.1%}"
                for t in tier_rev.index]
wedge_colors = [tier_colors.get(t, "#95a5a6") for t in tier_rev.index]
axes[1].pie(tier_rev.values, labels=wedge_labels, colors=wedge_colors,
            autopct="", startangle=90, textprops={"fontsize": 10})
axes[1].set_title("Revenue Share by Tier", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ── 2b. Tier migration over simulation epochs ─────────────────────
# Track cumulative breakout promotions (Tier 4 -> Tier 2) by epoch
if len(transitions) > 0:
    # Aggregate across runs: mean cumulative migrations per epoch
    migration_by_epoch = (
        transitions
        .groupby(["run_id", "epoch"])
        .size()
        .reset_index(name="migrations")
    )
    # Cumulative per run
    migration_by_epoch["cumulative"] = (
        migration_by_epoch
        .groupby("run_id")["migrations"]
        .cumsum()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: cumulative migrations per run
    for rid, grp in migration_by_epoch.groupby("run_id"):
        axes[0].step(grp["epoch"], grp["cumulative"], where="post",
                     label=f"Run {rid}", alpha=0.8, linewidth=2)
    axes[0].set_xlabel("Epoch (week)")
    axes[0].set_ylabel("Cumulative Tier 4 -> Tier 2 Migrations")
    axes[0].set_title("Breakout Tier Migrations Over Time", fontweight="bold")
    axes[0].legend()

    # Right: summary mean from summary.parquet
    axes[1].plot(summary["epoch"], summary["mean_breakout_success_count"],
                 color="#e74c3c", linewidth=2, label="Mean breakout successes")
    axes[1].fill_between(
        summary["epoch"],
        summary["ci_lower_breakout_success_count"],
        summary["ci_upper_breakout_success_count"],
        alpha=0.2, color="#e74c3c",
    )
    axes[1].set_xlabel("Epoch (week)")
    axes[1].set_ylabel("Cumulative Breakout Successes")
    axes[1].set_title("Breakout Success Count (mean +/- 95% CI)", fontweight="bold")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("No tier transitions recorded in this simulation.")

<a id='3'></a>
## 3. Revenue Convergence

In [ ]:
# ── 3a. Revenue convergence with confidence bands ─────────────────
metrics_cfg = [
    ("net_revenue",         "Net Revenue",         "#2c3e50"),
    ("total_revenue",       "Total Revenue",       "#2980b9"),
    ("recommended_revenue", "Recommended Revenue", "#27ae60"),
    ("organic_revenue",     "Organic Revenue",     "#f39c12"),
    ("discount_cost",       "Discount Cost",       "#e74c3c"),
]

fig, ax = plt.subplots(figsize=(16, 8))

epochs = summary["epoch"]

for key, label, color in metrics_cfg:
    mean_col = f"mean_{key}"
    std_col = f"std_{key}"
    mean = summary[mean_col] * SCALE
    std = summary[std_col] * SCALE

    ax.plot(epochs, mean / 1e6, color=color, linewidth=2, label=label)
    ax.fill_between(epochs, (mean - std) / 1e6, (mean + std) / 1e6,
                    alpha=0.12, color=color)

# Annotate convergence epoch for net revenue
conv_epoch = convergence.get("net_revenue", None)
if conv_epoch and conv_epoch <= NUM_EPOCHS:
    conv_val = summary.loc[summary["epoch"] == conv_epoch, "mean_net_revenue"].iloc[0] * SCALE / 1e6
    ax.axvline(conv_epoch, color="#7f8c8d", linestyle="--", alpha=0.7)
    ax.annotate(
        f"CV < 5% @ epoch {conv_epoch}",
        xy=(conv_epoch, conv_val), xytext=(conv_epoch + 2, conv_val * 1.08),
        fontsize=10, fontstyle="italic",
        arrowprops=dict(arrowstyle="->", color="#7f8c8d"),
    )
else:
    # Not yet converged
    ax.text(0.98, 0.02, "Net revenue has not converged (CV > 5%)",
            transform=ax.transAxes, ha="right", fontsize=10,
            fontstyle="italic", color="#e74c3c")

# Weekly target line
ax.axhline(WEEKLY_TARGET / 1e6, color="gray", linestyle=":", alpha=0.5)
ax.text(1, WEEKLY_TARGET / 1e6 * 1.01, "CVS weekly target ($57.1M)",
        fontsize=9, color="gray")

ax.set_xlabel("Epoch (week)")
ax.set_ylabel("Weekly Revenue ($M, scaled to 10M customers)")
ax.set_title("Revenue Convergence Across Monte Carlo Runs", fontweight="bold")
ax.legend(loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
plt.tight_layout()
plt.show()

In [ ]:
# ── 3b. Individual run overlay ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: net revenue per run
for rid, df in runs.items():
    axes[0].plot(df["epoch"], df["net_revenue"] * SCALE / 1e6,
                 alpha=0.6, linewidth=1.5, label=f"Run {rid}")
axes[0].plot(summary["epoch"], summary["mean_net_revenue"] * SCALE / 1e6,
             color="black", linewidth=2.5, label="Mean")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Net Revenue ($M)")
axes[0].set_title("Net Revenue by Run", fontweight="bold")
axes[0].legend(fontsize=8)

# Right: hit rate convergence
for rid, df in runs.items():
    axes[1].plot(df["epoch"], df["hit_rate_at_5"],
                 alpha=0.6, linewidth=1.5, label=f"Run {rid}")
axes[1].plot(summary["epoch"], summary["mean_hit_rate_at_5"],
             color="black", linewidth=2.5, label="Mean")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Hit Rate @5")
axes[1].set_title("Hit Rate @5 Convergence", fontweight="bold")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

<a id='4'></a>
## 4. Price Elasticity Landscape

In [ ]:
# ── 4a. Elasticity vs revenue, colored by tier ────────────────────
el_df = elasticity.merge(
    tiers_df[["product_id", "tier", "total_revenue"]],
    on="product_id", how="inner",
)

fig, ax = plt.subplots(figsize=(14, 8))

for t in sorted(el_df["tier"].unique()):
    subset = el_df[el_df["tier"] == t]
    ax.scatter(
        subset["elasticity_beta"],
        subset["total_revenue"] / 1e6,
        c=tier_colors.get(t, "#95a5a6"),
        label=f"Tier {t} ({len(subset):,})",
        alpha=0.4, s=15, edgecolors="none",
    )

# Highlight sweet spot: high elasticity (< -1.0) + low revenue = opportunity
sweet = el_df[(el_df["elasticity_beta"] < -1.0) & (el_df["total_revenue"] < el_df["total_revenue"].median())]
if len(sweet) > 0:
    ax.scatter(sweet["elasticity_beta"], sweet["total_revenue"] / 1e6,
               facecolors="none", edgecolors="black", s=60, linewidth=1.2,
               label=f"Sweet spot ({len(sweet):,})")

ax.set_xlabel("Elasticity (beta coefficient)")
ax.set_ylabel("Total Revenue ($M)")
ax.set_title("Price Elasticity vs Revenue by Tier\n(left = more price-sensitive)",
             fontweight="bold")
ax.legend()
ax.axvline(-1.0, color="gray", linestyle="--", alpha=0.4, label="Elasticity = -1")
plt.tight_layout()
plt.show()

print(f"Sweet spot: {len(sweet)} products with elasticity < -1.0 and below-median revenue")
print("These are the highest-opportunity products for discount-driven growth.")

In [ ]:
# ── 4b. Optimal discount curve for top 5 Tier 2 products ─────────
curves = pd.read_parquet(MODEL_DIR / "discount_response_curves.parquet")
tier2_top = (
    tiers_df[tiers_df["tier"] == 2]
    .nlargest(5, "total_revenue")
    .merge(curves, on="product_id")
)

discount_levels = [0, 5, 10, 15, 20, 25, 30]
score_cols = [f"score_at_{d}pct" for d in discount_levels]

fig, ax = plt.subplots(figsize=(12, 6))

for _, row in tier2_top.iterrows():
    pid = int(row["product_id"])
    pinfo = product_lookup.get(str(pid), {})
    name = pinfo.get("name", f"Product {pid}")[:40]
    scores = [float(row[c]) for c in score_cols]
    ax.plot(discount_levels, scores, marker="o", linewidth=2, label=name)

ax.set_xlabel("Discount Level (%)")
ax.set_ylabel("Predicted Score")
ax.set_title("Discount Response Curves: Top 5 Tier 2 Products", fontweight="bold")
ax.legend(fontsize=8, loc="best")
ax.set_xticks(discount_levels)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. Breakout Candidates

In [ ]:
# ── 5a. Breakout success timeline ────────────────────────────────
# For each promoted product, show the epoch it was promoted
if len(breakout_results) > 0:
    # Take run 0 as representative
    run0_breakouts = breakout_results[breakout_results["run_id"] == 0].copy()
    run0_breakouts = run0_breakouts.rename(columns={"promoted_epoch": "epoch"}).sort_values("epoch")

    # Merge with candidate info
    run0_breakouts = run0_breakouts.merge(
        breakout_candidates[["product_id", "product_name", "breakout_score",
                             "estimated_discount_to_break_in"]],
        on="product_id", how="left",
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Left: timeline of promotions
    promoted_epochs = run0_breakouts.groupby("epoch").size()
    cumulative = promoted_epochs.reindex(range(1, NUM_EPOCHS + 1), fill_value=0).cumsum()
    axes[0].step(cumulative.index, cumulative.values, where="post",
                 color="#2980b9", linewidth=2.5)
    axes[0].fill_between(cumulative.index, 0, cumulative.values,
                         alpha=0.15, color="#2980b9", step="post")
    for epoch, count in promoted_epochs.items():
        axes[0].annotate(
            f"+{count}", xy=(epoch, cumulative[epoch]),
            fontsize=8, ha="center", va="bottom",
        )
    axes[0].set_xlabel("Epoch (week)")
    axes[0].set_ylabel("Cumulative Promotions")
    axes[0].set_title("Breakout Promotion Timeline (Run 0)", fontweight="bold")
    axes[0].set_xlim(0, NUM_EPOCHS + 1)

    # Right: predicted vs actual discount comparison
    # Actual discount = what was offered from breakout_discounts array
    bp_df = breakout_candidates.copy()
    bp_df["actual_discount"] = breakout_discounts[:len(bp_df)]
    bp_df["promoted"] = bp_df["product_id"].isin(run0_breakouts["product_id"])

    colors = ["#27ae60" if p else "#e74c3c" for p in bp_df["promoted"]]
    axes[1].scatter(
        bp_df["estimated_discount_to_break_in"] * 100,
        bp_df["actual_discount"] * 100,
        c=colors, alpha=0.7, s=50, edgecolors="white", linewidth=0.5,
    )
    lims = [0, max(bp_df["actual_discount"].max(), bp_df["estimated_discount_to_break_in"].max()) * 100 + 2]
    axes[1].plot(lims, lims, "--", color="gray", alpha=0.5)
    axes[1].set_xlabel("Estimated Discount to Break In (%)")
    axes[1].set_ylabel("Actual Discount Offered (%)")
    axes[1].set_title("Predicted vs Actual Discount\n(green = promoted, red = not)",
                      fontweight="bold")

    plt.tight_layout()
    plt.show()

    # Success rate
    total_candidates = len(breakout_pids)
    promoted_count = len(run0_breakouts)
    print(f"\nBreakout success rate (Run 0): {promoted_count}/{total_candidates} "
          f"({promoted_count / total_candidates:.0%})")
else:
    print("No breakout results recorded.")

In [ ]:
# ── 5b. Cross-run breakout consistency ────────────────────────────
if len(breakout_results) > 0 and NUM_RUNS > 1:
    # Which products got promoted across all runs?
    promo_counts = (
        breakout_results
        .groupby("product_id")["run_id"]
        .nunique()
        .reset_index(name="runs_promoted")
        .sort_values("runs_promoted", ascending=False)
    )
    promo_counts = promo_counts.merge(
        breakout_candidates[["product_id", "product_name", "breakout_score"]],
        on="product_id", how="left",
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    run_dist = promo_counts["runs_promoted"].value_counts().sort_index()
    ax.bar(run_dist.index, run_dist.values, color="#2980b9", edgecolor="white")
    ax.set_xlabel(f"Number of Runs Promoted (out of {NUM_RUNS})")
    ax.set_ylabel("Number of Products")
    ax.set_title("Breakout Promotion Consistency Across MC Runs", fontweight="bold")
    ax.set_xticks(range(1, NUM_RUNS + 1))
    for bar_x, bar_y in zip(run_dist.index, run_dist.values):
        ax.text(bar_x, bar_y + 0.3, str(bar_y), ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Top 10 most consistent breakouts
    print(f"\nTop 10 most consistent breakout products (promoted in most runs):")
    display(promo_counts.head(10)[["product_id", "product_name", "breakout_score",
                                    "runs_promoted"]].reset_index(drop=True))
else:
    print("Only 1 run — cross-run analysis requires multiple MC runs.")

<a id='6'></a>
## 6. Comparison to CVS 10-K

In [ ]:
# ── 6a. Simulated vs 10-K benchmarks ─────────────────────────────
# CVS 10-K reference points (proportional to 10M of 74M ExtraCare)
TARGET_ANNUAL = 2.97e9
CVS_MARGIN = 0.29

avg_rev_per_visit = params.get("avg_revenue_per_visit", 35.0)
visits_per_year = (last["mean_total_revenue"] / avg_rev_per_visit) / SIM_CUSTOMERS * 52

benchmarks = pd.DataFrame({
    "Metric": [
        "Annual Revenue (10M customers)",
        "Weekly Revenue",
        "Discount Rate",
        "Gross Margin After Discounts",
        "Active Customer Rate",
        "Avg Visits/Customer/Year",
        "Avg Basket Size",
        "Coupon Redemption Rate (Hit@5)",
        "Coupons/Customer/Week",
    ],
    "Simulated": [
        f"${scaled_annual / 1e9:.2f}B",
        f"${scaled_weekly_net / 1e6:.1f}M",
        f"{discount_rate:.1%}",
        f"{gross_margin_after:.1%}",
        f"{active:.1%}",
        f"{visits_per_year:.1f}",
        f"${avg_rev_per_visit:.0f}",
        f"{hit5:.1%}",
        f"{coupons_per:.1f}",
    ],
    "CVS 10-K Target": [
        "~$2.97B",
        "~$57.1M",
        "< 10-12%",
        "~28-30%",
        "60-70%",
        "25-50",
        "$30-40",
        "15-25%",
        "3-8",
    ],
    "Verdict": [
        "PASS" if 2.0e9 <= scaled_annual <= 4.0e9 else "FAIL",
        "PASS" if abs(scaled_weekly_net - WEEKLY_TARGET) / WEEKLY_TARGET < 0.33 else "FAIL",
        "PASS" if discount_rate <= 0.12 else "FAIL",
        "PASS" if gross_margin_after >= 0.16 else "FAIL",
        "PASS" if 0.60 <= active <= 0.70 else ("WARN" if active >= 0.30 else "FAIL"),
        "PASS" if 25 <= visits_per_year <= 50 else ("WARN" if 15 <= visits_per_year <= 75 else "FAIL"),
        "PASS" if 30 <= avg_rev_per_visit <= 40 else "WARN",
        "PASS" if 0.15 <= hit5 <= 0.25 else "WARN",
        "PASS" if 3 <= coupons_per <= 8 else "WARN",
    ],
})

def _color_verdict(val):
    if val == "PASS":
        return "background-color: #d4edda"
    elif val == "WARN":
        return "background-color: #fff3cd"
    elif val == "FAIL":
        return "background-color: #f8d7da"
    return ""

print("=" * 80)
print("SIMULATION vs CVS 10-K FINANCIAL BENCHMARKS")
print("=" * 80)
display(
    benchmarks.style
    .hide(axis="index")
    .map(_color_verdict, subset=["Verdict"])
)

In [ ]:
# ── 6b. Revenue distribution sanity check ────────────────────────
tier_rev = tiers_df.groupby("tier")["total_revenue"].sum()
total_rev = tier_rev.sum()

cvs_targets = {
    1: (0.25, 0.35, "Top ~1%: core revenue drivers"),
    2: (0.30, 0.35, "Next ~15%: discount-responsive"),
    3: (0.20, 0.25, "Next ~30%: organic sellers"),
    4: (0.05, 0.15, "Bottom ~54%: long tail"),
}

fig, ax = plt.subplots(figsize=(12, 6))

tiers_present = sorted(tier_rev.index)
x = np.arange(len(tiers_present))
width = 0.35

sim_shares = [tier_rev[t] / total_rev for t in tiers_present]
target_mids = []
target_errs = []
for t in tiers_present:
    if t in cvs_targets:
        lo, hi, _ = cvs_targets[t]
        target_mids.append((lo + hi) / 2)
        target_errs.append((hi - lo) / 2)
    else:
        target_mids.append(0)
        target_errs.append(0)

ax.bar(x - width / 2, sim_shares, width, label="Simulated",
       color=[tier_colors.get(t, "#95a5a6") for t in tiers_present],
       edgecolor="white", linewidth=1.5)
ax.bar(x + width / 2, target_mids, width, yerr=target_errs,
       label="CVS 10-K Target", color="lightgray", edgecolor="#333",
       linewidth=1.5, capsize=5)

ax.set_xlabel("Product Tier")
ax.set_ylabel("Revenue Share")
ax.set_title("Revenue Distribution: Simulated vs CVS 10-K Target", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([f"Tier {t}" for t in tiers_present])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()

# Annotate actual values
for i, (share, t) in enumerate(zip(sim_shares, tiers_present)):
    ax.text(i - width / 2, share + 0.01, f"{share:.1%}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nDeviation discussion:")
print("- Tier 1 has only 50 products but captures a large revenue share because")
print("  these are the highest-velocity items in the catalog (e.g. baby care essentials).")
print("- No Tier 3 products exist in this tier assignment because the organic purchase")
print("  ratio threshold was not met by enough products with the current data generation.")
print("- The full-scale overnight run with more MC runs will produce tighter CI bands.")

<a id='7'></a>
## 7. Recommendation for Production

In [ ]:
# ── 7a. Top Tier 2 products for immediate coupon campaigns ───────
tier2_products = (
    tiers_df[tiers_df["tier"] == 2]
    .merge(elasticity[["product_id", "elasticity_beta", "optimal_discount",
                        "coupon_redemption_rate"]],
           on="product_id", how="left", suffixes=("", "_el"))
    .sort_values("discount_sensitivity_score")
)

# Most discount-sensitive Tier 2 products (best coupon ROI)
top_coupon_targets = tier2_products.head(20).copy()
top_coupon_targets["product_name"] = top_coupon_targets["product_id"].apply(
    lambda pid: product_lookup.get(str(pid), {}).get("name", f"Product {pid}")[:50]
)

print("=" * 80)
print("TOP 20 TIER 2 PRODUCTS FOR IMMEDIATE COUPON CAMPAIGNS")
print("(Ranked by discount sensitivity — most responsive to coupons)")
print("=" * 80)
display(
    top_coupon_targets[
        ["product_id", "product_name", "total_revenue", "discount_sensitivity_score",
         "elasticity_beta", "optimal_discount", "repeat_purchase_rate"]
    ]
    .reset_index(drop=True)
    .style.format({
        "total_revenue": "${:,.0f}",
        "discount_sensitivity_score": "{:.3f}",
        "elasticity_beta": "{:.3f}",
        "optimal_discount": "{:.1%}",
        "repeat_purchase_rate": "{:.1%}",
    })
)

In [ ]:
# ── 7b. Best breakout bets from Tier 4 ────────────────────────────
# Products that were promoted across the most runs = highest confidence
if len(breakout_results) > 0:
    promo_confidence = (
        breakout_results
        .groupby("product_id")["run_id"]
        .nunique()
        .reset_index(name="runs_promoted")
    )
    best_bets = (
        breakout_candidates
        .merge(promo_confidence, on="product_id", how="left")
    )
    best_bets["runs_promoted"] = best_bets["runs_promoted"].fillna(0).astype(int)
    best_bets = best_bets.sort_values(
        ["runs_promoted", "breakout_score"], ascending=[False, False]
    )

    print("=" * 80)
    print("TOP 15 BREAKOUT BETS (Tier 4 -> Tier 2 candidates)")
    print(f"(Ranked by promotion consistency across {NUM_RUNS} MC runs)")
    print("=" * 80)
    display(
        best_bets.head(15)[
            ["product_id", "product_name", "category", "price", "breakout_score",
             "cosine_to_tier1", "estimated_discount_to_break_in", "runs_promoted"]
        ]
        .reset_index(drop=True)
        .style.format({
            "price": "${:.2f}",
            "breakout_score": "{:.3f}",
            "cosine_to_tier1": "{:.4f}",
            "estimated_discount_to_break_in": "{:.1%}",
        })
    )
else:
    print("No breakout data available.")

In [ ]:
# ── 7c. Discount budget and ROI projection ────────────────────────
weekly_disc_scaled = last["mean_discount_cost"] * SCALE
weekly_rec_rev_scaled = last["mean_recommended_revenue"] * SCALE
weekly_net_scaled = last["mean_net_revenue"] * SCALE
annual_disc = weekly_disc_scaled * 52
annual_rec_rev = weekly_rec_rev_scaled * 52
annual_net = weekly_net_scaled * 52

# ROI = incremental revenue from recommendations / discount spend
roi = weekly_rec_rev_scaled / weekly_disc_scaled if weekly_disc_scaled > 0 else 0

# Breakout-specific budget
avg_breakout_disc = float(breakout_discounts.mean()) if len(breakout_discounts) > 0 else 0
avg_breakout_price = float(
    breakout_candidates["price"].mean()
) if len(breakout_candidates) > 0 else 0
breakout_weekly_cost = avg_breakout_disc * avg_breakout_price * len(breakout_pids) * SCALE / SIM_CUSTOMERS

budget = pd.DataFrame({
    "Component": [
        "Total Weekly Discount Budget",
        "Annual Discount Budget",
        "Weekly Recommended Revenue",
        "Annual Recommended Revenue",
        "Recommendation ROI",
        "Annual Net Revenue (after discounts)",
        "Breakout Campaign Weekly Cost (est.)",
        "Avg Breakout Discount Level",
    ],
    "Value": [
        f"${weekly_disc_scaled / 1e6:.1f}M",
        f"${annual_disc / 1e6:.0f}M",
        f"${weekly_rec_rev_scaled / 1e6:.1f}M",
        f"${annual_rec_rev / 1e9:.2f}B",
        f"{roi:.1f}x",
        f"${annual_net / 1e9:.2f}B",
        f"${breakout_weekly_cost / 1e3:.0f}K",
        f"{avg_breakout_disc:.1%}",
    ],
})

print("=" * 80)
print("DISCOUNT BUDGET & ROI PROJECTION (scaled to 10M customers)")
print("=" * 80)
display(budget.style.hide(axis="index"))

print(f"\nFor every $1 spent on coupons, the model generates ${roi:.1f} in")
print(f"recommended-channel revenue. The discount budget of ~${annual_disc / 1e6:.0f}M/year")
print(f"represents {discount_rate:.1%} of total revenue, well within the CVS front store")
print(f"gross margin of ~29%.")

In [ ]:
# ── 7d. Summary visualization ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Revenue waterfall
components = ["Organic", "Recommended", "Discounts", "Net"]
values = [
    last["mean_organic_revenue"] * SCALE / 1e6,
    last["mean_recommended_revenue"] * SCALE / 1e6,
    -last["mean_discount_cost"] * SCALE / 1e6,
    last["mean_net_revenue"] * SCALE / 1e6,
]
colors_wf = ["#27ae60", "#2980b9", "#e74c3c", "#2c3e50"]
axes[0].bar(components, values, color=colors_wf, edgecolor="white", linewidth=1.5)
for i, (comp, val) in enumerate(zip(components, values)):
    axes[0].text(i, val + (0.3 if val > 0 else -0.8),
                 f"${val:.1f}M", ha="center", fontsize=9, fontweight="bold")
axes[0].set_title("Weekly Revenue Waterfall ($M)", fontweight="bold")
axes[0].set_ylabel("$M per week")

# Hit rate over time
axes[1].plot(summary["epoch"], summary["mean_hit_rate_at_5"],
             color="#2980b9", linewidth=2, label="Hit@5")
axes[1].plot(summary["epoch"], summary["mean_hit_rate_at_2"],
             color="#e67e22", linewidth=2, label="Hit@2")
axes[1].fill_between(summary["epoch"],
                     summary["ci_lower_hit_rate_at_5"],
                     summary["ci_upper_hit_rate_at_5"],
                     alpha=0.15, color="#2980b9")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_xlabel("Epoch")
axes[1].set_title("Hit Rate Over Time", fontweight="bold")
axes[1].legend()

# Active customer % over time
axes[2].plot(summary["epoch"], summary["mean_active_customer_pct"] * 100,
             color="#8e44ad", linewidth=2)
axes[2].fill_between(summary["epoch"],
                     summary["ci_lower_active_customer_pct"] * 100,
                     summary["ci_upper_active_customer_pct"] * 100,
                     alpha=0.15, color="#8e44ad")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Active %")
axes[2].set_title("Active Customer Rate Over Time", fontweight="bold")

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("PRODUCTION RECOMMENDATIONS")
print("=" * 80)
print(f"1. Launch coupon campaigns for the top 20 Tier 2 products above.")
print(f"   Expected discount budget: ~${annual_disc / 1e6:.0f}M/year at {discount_rate:.1%} rate.")
print(f"2. Monitor {len(breakout_pids)} breakout candidates with ~{avg_breakout_disc:.0%} avg discount.")
print(f"   {breakout_n:.0f} products are already showing Tier 2-level traction.")
print(f"3. Retrain the model every {params['retrain_interval']} weeks on blended data.")
print(f"4. Run full-scale simulation ({TARGET_CUSTOMERS / 1e6:.0f}M customers, 50+ MC runs)")
print(f"   for production-grade confidence intervals.")